# HW3 

In [1]:
#!pip install openmeteo_requests

In [2]:
import openmeteo_requests
from datetime import datetime

## Iterator classes

In [3]:
class IncreaseSpeed:
    """
    Iterator for increasing speed step by step.
    """
    def __init__(self, current_speed: int, max_speed: int, step: int = 10):
        self.current = current_speed
        self.max_speed = max_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current + self.step > self.max_speed:
            raise StopIteration
        self.current += self.step
        return self.current

In [4]:
class DecreaseSpeed:
    """
    Iterator for decreasing speed step by step.
    """
    def __init__(self, current_speed: int, min_speed: int = 0, step: int = 10):
        self.current = current_speed
        self.min_speed = min_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current - self.step < self.min_speed:
            raise StopIteration
        self.current -= self.step
        return self.current

## Car class

In [5]:
class Car:
    _total_cars_on_road = 0

    def __init__(self, max_speed: int, current_speed: int = 0):
        self.max_speed = max_speed
        self.current_speed = current_speed
        # Car is considered on the road if its speed > 0
        self.state = current_speed > 0
        if self.state:
            Car._total_cars_on_road += 1

    # --- regular methods ---
    def accelerate(self, upper_border=None, step=10):
        """Increase speed, optionally to a given border."""
        old_speed = self.current_speed

        if not self.state:                     # car is parked
            self.state = True
            Car._total_cars_on_road += 1
            target = upper_border if upper_border is not None else step
            if target > self.max_speed:
                target = self.max_speed
            self.current_speed = target
            print(f"INFO: The speed of this car has been increased from {old_speed} to {self.current_speed}")
            return
        
        inc = IncreaseSpeed(self.current_speed, self.max_speed, step)  # Car on road
        if upper_border is not None:
            # gradually increase up to upper_border
            for new_speed in inc:
                if new_speed > upper_border:
                    break
                self.current_speed = new_speed
                print(f"INFO: Speed increases by {step}")
            if self.current_speed < upper_border:   # ensure final speed == upper_border
                self.current_speed = upper_border
        else:
            try:            # increase by one step
                self.current_speed = next(inc)
                print(f"INFO: Speed increases by {step}")
            except StopIteration:
                pass
        print(f"INFO: The speed of this car has been increased from {old_speed} to {self.current_speed}")

    def brake(self, lower_border=None, step=10):
        """Decrease speed, optionally to a given border."""
        if not self.state:
            print("INFO: Car is already parked.")
            return

        old_speed = self.current_speed
        dec = DecreaseSpeed(self.current_speed, min_speed=0, step=step)
        if lower_border is not None:
            for new_speed in dec:
                if new_speed < lower_border:
                    break
                self.current_speed = new_speed
                print(f"INFO: Speed decreases by {step}")
            if self.current_speed > lower_border:
                self.current_speed = max(lower_border, 0)
        else:
            try:
                self.current_speed = next(dec)
                print(f"INFO: Speed decreases by {step}")
            except StopIteration:
                pass
        print(f"INFO: The speed of this car has been decreased from {old_speed} to {self.current_speed}")


    def parking(self):
        """Take the car off the road (speed must be 0)."""
        if not self.state:
            print("INFO: Car is already parked.")
            return
        # bring speed to 0 if needed
        if self.current_speed > 0:
            self.brake(lower_border=0)
        else:
            print(f"INFO: The speed of this car has been decreased from {self.current_speed} to {self.current_speed}")
        self.state = False
        Car._total_cars_on_road -= 1
        print("Parking the car...")

    @classmethod
    def total_cars(cls):
        """Return the total number of cars currently on the road."""
        return cls._total_cars_on_road

    @staticmethod
    def show_weather():
        """Fetch and display current weather in St. Petersburg."""
        openmeteo = openmeteo_requests.Client()
        url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": 59.9386,
            "longitude": 30.3141,
            "current": ["temperature_2m", "apparent_temperature", "rain", "wind_speed_10m"],
            "wind_speed_unit": "ms",
            "timezone": "Europe/Moscow"
        }
        try:
            response = openmeteo.weather_api(url, params=params)[0]
            current = response.Current()
            temp = current.Variables(0).Value()
            apparent_temp = current.Variables(1).Value()
            rain = current.Variables(2).Value()
            wind = current.Variables(3).Value()

            print(f"Current temperature: {round(temp, 0)} C")
            print(f"Current apparent_temperature: {round(apparent_temp, 0)} C")
            print(f"Current rain: {rain} mm")
            print(f"Current wind_speed: {round(wind, 1)} m/s")
        except Exception as e:
            print(f"Weather service error: {e}")

## Testing with the provided example

In [6]:
car1 = Car(100, 20) # max_speed = 100, initial speed = 5
car2 = Car(60, 30) # max_speed = 60, initial speed = 30
car3 = Car(100, 0) # a car that is off road upon creation
print(f"Total cars on road: {Car.total_cars()}")

Total cars on road: 2


In [7]:
car1.accelerate(100)

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 20 to 100


In [8]:
car2.accelerate(50)

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 30 to 50


In [9]:
print("Speed of car 1:", car1.current_speed)

Speed of car 1: 100


In [10]:
print("Speed of car 2:", car2.current_speed)

Speed of car 2: 50


In [11]:
car1.brake(10)

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 100 to 10


In [12]:
car2.brake(0)
print("Total cars on road:", Car.total_cars())
car2.parking()
print("Total cars on road:", Car.total_cars())

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 50 to 0
Total cars on road: 2
INFO: The speed of this car has been decreased from 0 to 0
Parking the car...
Total cars on road: 1


In [13]:
car3.accelerate(80)# car3 is now on the road
car3.show_weather()
print("Total cars on road:", Car.total_cars())

INFO: The speed of this car has been increased from 0 to 80
Current temperature: 6.0 C
Current apparent_temperature: 1.0 C
Current rain: 0.0 mm
Current wind_speed: 5.9 m/s
Total cars on road: 2


In [14]:
car2.accelerate(10) # # car2 goes from parking on the road
print("Total cars on road:", Car.total_cars())

INFO: The speed of this car has been increased from 0 to 10
Total cars on road: 3


In [15]:
Car.show_weather()

Current temperature: 6.0 C
Current apparent_temperature: 1.0 C
Current rain: 0.0 mm
Current wind_speed: 5.9 m/s
